In [7]:
"""
Uplift Rule Search — 250-500 Loyalty Points Arm
=================================================
Finds 5-variable AND-rules that maximise uplift on the 250-500 merged loyalty
arm. Rules MUST contain: recency, monetary_value, length_of_relationship,
online_sales, retail_sales.

Constraints:
- 20% <= targeting share <= 50%
- >= 5,000 customers in the *treated* part of the targeted split
- >= 200 customers per group (control / treatment)

Output: top 10 ranked CSV + printed top-10 table.

Caveat: rules are ranked on observed uplift in the same data used to discover
them, so the top-of-list uplifts are optimistic. For thesis-grade reporting
hold out a validation split.
"""
from __future__ import annotations

from itertools import combinations, product
from pathlib import Path

import numpy as np
import pandas as pd

# ── Data loader (inlined; no external module dependency) ──────────────────────
FILE_PATH = r"Data/covariates_modeling_uplift_models_2026-03-13.csv"

DROP_INDICATORS = [
    "BNLX_ChurnP_SKUd_test_export.csv",
    "BNLX_ChurnP_SKUd_controle_export.csv",
    "BNLX_ChurnP_niks_test_export.csv",
    "BNLX_ChurnP_niks_controle_export.csv",
]

TREATMENT_CONVERTER = {
    "BNLX_ChurnP_10_test_export.csv":       "treatment_1",
    "BNLX_ChurnP_10_controle_export.csv":    "control_1",
    "BNLX_ChurnP_25_test_export.csv":        "treatment_2",
    "BNLX_ChurnP_25_controle_export.csv":    "control_2",
    "BNLX_ChurnP_5eu_test_export.csv":       "treatment_3",
    "BNLX_ChurnP_5eu_controle_export.csv":   "control_3",
    "BNLX_ChurnP_10eu_test_export.csv":      "treatment_4",
    "BNLX_ChurnP_10eu_controle_export.csv":  "control_4",
    "BNLX_ChurnP_250_test_export.csv":       "treatment_5",
    "BNLX_ChurnP_250_controle_export.csv":   "control_5",
    "BNLX_ChurnP_500_test_export.csv":       "treatment_6",
    "BNLX_ChurnP_500_controle_export.csv":   "control_6",
    "BNLX_ChurnP_SKUe_test_export.csv":      "treatment_7",
    "BNLX_ChurnP_SKUe_controle_export.csv":  "control_7",
}

CAT_COLS = ["has_rfl", "gender", "country_sk"]
NUM_COLS = [
    "recency", "monetary_value", "aov_per_customer",
    "length_of_relationship", "online_sales", "retail_sales",
    "food_total", "vhms_total", "sports_total", "beauty_total",
]


def coerce_metrics_to_numeric(df, cols):
    df = df.copy()
    df[cols] = df[cols].replace({",": ""}, regex=True).apply(pd.to_numeric, errors="coerce")
    return df


def load_data(path: str = FILE_PATH) -> pd.DataFrame:
    df = pd.read_csv(path)
    df = df[~df["treatment_indicator"].isin(DROP_INDICATORS)]
    df["treatment"] = df["treatment_indicator"].map(TREATMENT_CONVERTER)

    # Merge 250 (k=5) and 500 (k=6) loyalty arms into a single combined arm (k=5)
    df["treatment"] = df["treatment"].replace({
        "treatment_6": "treatment_5",
        "control_6":   "control_5",
    })

    df["experiment_k"] = df["treatment"].str.extract(r"(\d+)$").astype(int)
    df["gender"] = df["gender"].fillna("no_gender")
    source_nums = [c for c in NUM_COLS if c in df.columns]
    df = coerce_metrics_to_numeric(df, source_nums)
    df["aov_per_customer"] = np.where(
        df["frequency"] > 0, (df["monetary_value"] / df["frequency"]).round(0), 0)
    df[NUM_COLS] = df[NUM_COLS].fillna(0).astype("int64")
    df[CAT_COLS] = df[CAT_COLS].astype("object")
    return df


OUTPUT_DIR = Path("Output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Config ────────────────────────────────────────────────────────────────────
ARM_K              = 5                        # merged 250-500 loyalty arm
RULE_VARS          = ["recency", "monetary_value", "length_of_relationship",
                      "online_sales", "retail_sales"]
QUANTILE_SPLITS    = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
SHARE_MIN          = 0.20
SHARE_MAX          = 0.50
MIN_N_TREATED      = 5_000                    # >=5k in the *treated* arm of the targeted split
MIN_N_PER_GROUP    = 5_000                      # still need both groups present
MIN_UPLIFT_PP      = None                     # no floor; just rank top 10
REQUIRED_VARS      = {"recency", "monetary_value", "length_of_relationship",
                      "online_sales", "retail_sales"}
TOP_N_PRINT        = 10
TOP_N_EXPORT       = 10
RULE_SIZE          = 5                        # number of conditions per rule


# ── Step 1: filter to the 250-500 arm ─────────────────────────────────────────
def filter_arm(df: pd.DataFrame, arm_k: int) -> pd.DataFrame:
    sub = df[df["experiment_k"] == arm_k].copy()
    sub["group"] = np.where(sub["treatment"].str.startswith("treatment"),
                            "Treatment", "Control")
    return sub


# ── Step 2: build candidate split points per variable ─────────────────────────
def build_split_grid(df: pd.DataFrame, variables: list[str],
                     quantiles: list[float]) -> dict[str, list[float]]:
    grid = {}
    for v in variables:
        qs = df[v].quantile(quantiles).round(2).unique().tolist()
        grid[v] = sorted(qs)
    return grid


# ── Step 3: enumerate all k-variable AND-rules ────────────────────────────────
def enumerate_rules(split_grid: dict[str, list[float]],
                    rule_size: int) -> list[tuple]:
    """
    Each rule is a tuple of `rule_size` (var, op, threshold) conditions
    joined by AND. op is either "<=" or ">".
    """
    rules = []
    var_combos = list(combinations(split_grid.keys(), rule_size))
    for vs in var_combos:
        threshold_lists = [split_grid[v] for v in vs]
        op_lists = [["<=", ">"]] * rule_size
        for thresholds in product(*threshold_lists):
            for ops in product(*op_lists):
                rules.append(tuple(
                    (v, o, t) for v, o, t in zip(vs, ops, thresholds)
                ))
    return rules


# ── Step 4: evaluate one rule on the dataframe ────────────────────────────────
def apply_rule(df: pd.DataFrame, rule: tuple) -> pd.Series:
    """Return boolean mask for rows matching the rule."""
    mask = pd.Series(True, index=df.index)
    for var, op, thr in rule:
        col = df[var]
        mask &= (col <= thr) if op == "<=" else (col > thr)
    return mask


def evaluate_rule(df: pd.DataFrame, rule: tuple, total_n: int) -> dict | None:
    # Skip rules that don't contain all required variables
    rule_vars = {var for var, _, _ in rule}
    if not REQUIRED_VARS.issubset(rule_vars):
        return None

    mask = apply_rule(df, rule)
    n_targeted = int(mask.sum())
    n_share = n_targeted / total_n

    if not (SHARE_MIN <= n_share <= SHARE_MAX):
        return None

    targeted = df[mask]
    g = targeted.groupby("group")["reactivated"].agg(["sum", "count"])
    if "Control" not in g.index or "Treatment" not in g.index:
        return None
    n_c, n_t = int(g.loc["Control", "count"]), int(g.loc["Treatment", "count"])
    if n_c < MIN_N_PER_GROUP or n_t < MIN_N_PER_GROUP:
        return None
    if n_t < MIN_N_TREATED:
        return None

    rr_c = g.loc["Control", "sum"] / n_c
    rr_t = g.loc["Treatment", "sum"] / n_t
    uplift = rr_t - rr_c

    # 95% CI on the difference of two proportions (Wald)
    se = np.sqrt(rr_c * (1 - rr_c) / n_c + rr_t * (1 - rr_t) / n_t)
    ci_lo, ci_hi = uplift - 1.96 * se, uplift + 1.96 * se

    return {
        "rule":         _rule_str(rule),
        "n_targeted":   n_targeted,
        "share":        round(n_share, 4),
        "n_ctrl":       n_c,
        "n_trt":        n_t,
        "ctrl_rr":      round(rr_c, 5),
        "trt_rr":       round(rr_t, 5),
        "uplift_pp":    round(uplift * 100, 3),
        "ci_lo_pp":     round(ci_lo * 100, 3),
        "ci_hi_pp":     round(ci_hi * 100, 3),
        "significant":  ci_lo > 0,
    }


def _rule_str(rule: tuple) -> str:
    parts = [f"{v} {op} {thr:g}" for v, op, thr in rule]
    return " AND ".join(parts)


# ── Step 5: full search loop ──────────────────────────────────────────────────
def search_rules(df: pd.DataFrame) -> pd.DataFrame:
    total_n = len(df)
    grid = build_split_grid(df, RULE_VARS, QUANTILE_SPLITS)
    rules = enumerate_rules(grid, RULE_SIZE)
    print(f"  Variables       : {RULE_VARS}")
    print(f"  Required in rule: {sorted(REQUIRED_VARS)}")
    print(f"  Splits per var  : {[len(grid[v]) for v in RULE_VARS]}")
    print(f"  Candidate rules : {len(rules):,}")

    results = []
    for r in rules:
        res = evaluate_rule(df, r, total_n)
        if res is not None:
            results.append(res)

    if not results:
        return pd.DataFrame()
    out = pd.DataFrame(results).sort_values("uplift_pp", ascending=False).reset_index(drop=True)
    return out


# ── Step 6: format and print ──────────────────────────────────────────────────
def print_top(ranked: pd.DataFrame, n: int) -> None:
    if ranked.empty:
        print("  No rules met the criteria.")
        return
    cols = ["rule", "n_targeted", "share", "n_ctrl", "n_trt",
            "ctrl_rr", "trt_rr", "uplift_pp", "ci_lo_pp", "ci_hi_pp", "significant"]
    print(f"\n  Top {n} rules by uplift:")
    print(ranked[cols].head(n).to_string(index=True))


# ── Main ──────────────────────────────────────────────────────────────────────
def main() -> None:
    print("Loading data...")
    df_full = load_data()
    df_arm = filter_arm(df_full, ARM_K)
    n_c = (df_arm["group"] == "Control").sum()
    n_t = (df_arm["group"] == "Treatment").sum()
    print(f"  Arm k={ARM_K} (250-500 loyalty merged): "
          f"{len(df_arm):,} rows, {n_c:,} control, {n_t:,} treatment\n")

    print("Searching rules...")
    ranked_all = search_rules(df_arm)
    print(f"  Rules meeting all filters: {len(ranked_all):,}\n")

    print_top(ranked_all, TOP_N_PRINT)

    out_path = OUTPUT_DIR / "uplift_rules_loyalty_250_500.csv"
    ranked_all.head(TOP_N_EXPORT).to_csv(out_path, index=False)
    print(f"\n  Exported top {min(len(ranked_all), TOP_N_EXPORT)} rules → {out_path}")
    print("\n  Caveat: uplifts ranked on the same data used to discover rules; "
          "expect mild upward bias. Significant=True means 95% CI excludes 0.")


if __name__ == "__main__":
    main()

Loading data...


C:\Users\tsterk\AppData\Local\Temp\ipykernel_22380\2415237928.py:69: DtypeWarning: Columns (0: monetary_value, 1: total_volume, 2: online_sales, 3: retail_sales, 4: food_total, 5: vhms_total, 6: sports_total, 7: beauty_total, 8: monetary_value_52wk, 9: online_sales_52w, 10: retail_sales_52w, 11: monetary_value_53w_104w, 12: online_sales_53w_104w, 13: retail_sales_53w_104w) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


  Arm k=5 (250-500 loyalty merged): 60,685 rows, 30,254 control, 30,431 treatment

Searching rules...
  Variables       : ['recency', 'monetary_value', 'length_of_relationship', 'online_sales', 'retail_sales']
  Required in rule: ['length_of_relationship', 'monetary_value', 'online_sales', 'recency', 'retail_sales']
  Splits per var  : [9, 9, 9, 3, 8]
  Candidate rules : 559,872
  Rules meeting all filters: 16,090


  Top 10 rules by uplift:
                                                                                                                        rule  n_targeted   share  n_ctrl  n_trt  ctrl_rr   trt_rr  uplift_pp  ci_lo_pp  ci_hi_pp  significant
0   recency <= 1027 AND monetary_value <= 99 AND length_of_relationship <= 1668 AND online_sales <= 55 AND retail_sales > 15       13332  0.2197    6596   6736  0.02198  0.03058      0.860     0.317     1.402         True
1    recency <= 1027 AND monetary_value <= 99 AND length_of_relationship <= 1668 AND online_sales <= 0 AND ret